## Issues
1. Capacity availability on certain date
   1. #of trucks available
2. Look at the transmode
   1. (Truck - Container) 40Ft truck  
3. Convert (shipment_plans) Units into pallets
4. Container should be limited by weight, area utilization, Volume
   1. Add-on: Conditions are Configurable from the user
   2. Calculate golden ratio (Currently A -> B)
5. Pull-in method


# Code

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time


## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position
import uuid

##
from utils.visualize import container_visualization
from utils.measure_conversion import *

logger = getLogger("load_planner")


# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [4]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [5]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [26]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Loading Data
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_demand_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


D:\acies_solutions\solutions-inventory-optimization\database\helper.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


## Create necessary features, calculations




In [35]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [47]:
# Step 1 Build_shipment_candidates

def build_shipment_candidates(
    shipment_demand_df,
    sku_uom_df
):
    """
    Build shipment candidate table.

    Returns
    -------
    shipment_candidate_df
    """

    sku_uom_df['sku_id'] = sku_uom_df['sku_id'].astype(str)
    shipment_demand_df['sku_id'] = shipment_demand_df['sku_id'].astype(str)

    shipment_candidate_df = (
        shipment_demand_df
        .merge(
            sku_uom_df,
            on="sku_id",
            how="left"
        )
    )

    ### Filter for the ones that are divisible
    shipment_candidate_df = (
        shipment_candidate_df
        [
            shipment_candidate_df[
                "unit_count_in_pallet"
            ]
            > 0
        ]
    )


    shipment_candidate_df[
        "required_pallets"
    ] = (
        shipment_candidate_df[
            "planned_quantity"
        ]
        /
        shipment_candidate_df[
            "unit_count_in_pallet"
        ]
    )

    shipment_candidate_df[
        "full_pallets"
    ] = (

        shipment_candidate_df[
            "required_pallets"
        ]
        .apply(np.floor)
        .astype(int)
    )

    shipment_candidate_df[
        "remaining_units"
    ] = (

        shipment_candidate_df[
            "planned_quantity"
        ]
        %
        shipment_candidate_df[
            "unit_count_in_pallet"
        ]
    )

    shipment_candidate_df[
        "partial_fill_pct"
    ] = (

        shipment_candidate_df[
            "remaining_units"
        ]
        /
        shipment_candidate_df[
            "unit_count_in_pallet"
        ]
    )

    return shipment_candidate_df


In [48]:
shipment_candidate_df = (
    build_shipment_candidates(
        shipment_demand_df=shipment_demand_df,
        sku_uom_df=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', 'pallet_weight_in_kg', 'item_weight_in_kg']]
    )
)

In [49]:
shipment_candidate_df.head()

,order_line_id,shipment_id,sku_id,actual_delivery_date,origin_location_id,destination_location_id,estimated_delivery_date,planned_quantity,shipped_quantity,weight_kg,...,unit_count_in_pallet,pallet_height_mm,pallet_width_mm,pallet_length_mm,pallet_weight_in_kg,item_weight_in_kg,required_pallets,full_pallets,remaining_units,partial_fill_pct
0,23654,Lane_From_6011_To_0848-0001-140201-13-Apr-26-3,140201,None,6011,0848,2026-04-13,4320,0,1675387.41,...,2160.0,1155.0,1016.0,1219.0,387.82,0.18,2.0,2,0.0,0.0
1,23655,Lane_From_6011_To_0848-0001-128489-12-Apr-26-1,128489,None,6011,0848,2026-04-12,12960,0,6924934.63,...,720.0,1190.0,1016.0,1219.0,534.33,0.74,18.0,18,0.0,0.0
2,23656,Lane_From_6011_To_0848-0001-128472-14-Apr-26-1,128472,None,6011,0848,2026-04-14,36720,0,13574557.07,...,2160.0,1057.0,1016.0,1219.0,369.68,0.17,17.0,17,0.0,0.0
3,23657,Lane_From_6011_To_0848-0001-140249-14-Apr-26-1,140249,None,6011,0848,2026-04-14,28080,0,10890018.17,...,2160.0,1155.0,1016.0,1219.0,387.82,0.18,13.0,13,0.0,0.0
4,23658,Lane_From_6011_To_0848-0001-128685-15-Apr-26-1,128685,None,6011,0848,2026-04-15,17280,0,9233246.18,...,720.0,1190.0,1016.0,1219.0,534.33,0.74,24.0,24,0.0,0.0


In [ ]:
## 


pallet_candidate_df = pd.DataFrame([], columns=[
	"candidate_pallet_id",
	"shipment_id",
	"sku_id",
	"destination_location_id",
	"estimated_delivery_date",
	"priority",
	"units_in_pallet",
	"weight_in_kg",
	"floor_area_m2",
	"volume_m3",
	"is_partial_pallet",
	"fill_pct"
])


In [ ]:

container_build_df = pd.DataFrame([], columns=[
    "container_id",
    "candidate_pallet_id",
    "sku_id",
    "weight_in_kg",
    "position_x",
    "position_y",
    "position_z",
    "axle_zone",
    "load_sequence"
])


In [ ]:

pending_load_df = pd.DataFrame([], columns=[
    "candidate_pallet_id",
    "shipment_id",
    "sku_id",
    "remaining_units",
    "reason"
])


# Optimizer Flow V2

- STEP 1:
build_shipment_candidates()

- STEP 2:
expand_pallet_candidates()

- STEP 3:
initialize_container()

- STEP 4:
build_container_load()

```
    while capacity exists:
        find_next_position()
        calculate_axle_loads()
        validate_constraints()
        load pallet
```
- STEP 5
build_pending_loads()

- STEP 6
build_container_objects()

- STEP 7
visualize()

In [31]:
container_state = {
    "used_weight_kg":0,
    "used_floor_area_m2":0,
    "used_volume_m3":0,

    "front_axle_load":0,
    "rear_axle_load":0,

    "current_x":0,
    "current_z":0,

    "row_depth":0
}

In [ ]:

shipment_candidate_df[
    [
        "sku_id",
        "planned_quantity",
        "unit_count_in_pallet",
        "required_pallets",
        "full_pallets",
        "remaining_units",
        "partial_fill_pct"
    ]
].head()

In [65]:
import uuid

from objects.PalletCandidate import PalletCandidate


def expand_pallet_candidates(
        shipment_candidate_df,
        partial_pallet_threshold=0.80
):

    pallet_candidates = []

    for _, row in shipment_candidate_df.iterrows():

        # --------------------------
        # Full Pallets
        # --------------------------

        for _ in range(
                int(row["full_pallets"])
        ):

            unitsInPallet=int(row["unit_count_in_pallet"])
            isPartialPallet=False
            weightIn_kg = float(row["pallet_weight_in_kg"])
            fillPct = 1.0

            _pallet_candidate_ = PalletCandidate(
                candidatePalletId=f"PALLET_{uuid.uuid4().hex[:12].upper()}",
                shipmentId=str(row["shipment_id"]),
                skuId=str(row["sku_id"]),
                originLocationId=str(row["origin_location_id"]),
                destinationLocationId=str(row["destination_location_id"]),
                estimatedDeliveryDate=row["estimated_delivery_date"],
                priority=int(row["priority"]),
                serviceLevel=int(row["service_level"]),
                unitsInPallet=unitsInPallet, # partial pallet / full pallet
                isPartialPallet=isPartialPallet, # partial pallet / full pallet
                fillPct=fillPct, # partial pallet / full pallet
                weightIn_kg=weightIn_kg, # partial pallet / full pallet
                floorArea_m2=(
                    row["pallet_length_mm"]
                    *
                    row["pallet_width_mm"]
                ) / 1_000_000,

                volume_m3=(
                    row["pallet_length_mm"]
                    *
                    row["pallet_width_mm"]
                    *
                    row["pallet_height_mm"]
                ) / 1_000_000_000,

                dimensions={
                    'depth':int(
                        row["pallet_length_mm"]
                    ),
                    'width':int(
                        row["pallet_width_mm"]
                    ),
                    'height':int(
                        row["pallet_height_mm"]
                    )
                },

                position={
                    'x':0,
                    'y':0,
                    'z':0
                },

                label=f"SKU {row['sku_id']}"
            )
            pallet_candidates.append(_pallet_candidate_)


            # --------------------------
            # Partial Pallet
            # --------------------------

            if (
                    row["partial_fill_pct"]
                    >=
                    partial_pallet_threshold
            ):
                fillPct = float(
                    row["partial_fill_pct"]
                )
                isPartialPallet=True
                unitsInPallet=int(row["remaining_units"])
                weightIn_kg *= weightIn_kg

                _pallet_candidate_.fillPct = fillPct
                _pallet_candidate_.isPartialPallet = isPartialPallet
                _pallet_candidate_.unitsInPallet = unitsInPallet
                _pallet_candidate_.weightIn_kg = weightIn_kg
                pallet_candidates.append(_pallet_candidate_)

    return pallet_candidates


In [66]:
pallet_candidate_df = (
    expand_pallet_candidates(
        shipment_candidate_df=shipment_candidate_df
    )
)

print(
    len(
        pallet_candidate_df
    )
)

# pallet_candidate_df.head()

304315


## Phase 1:



In [ ]:
def build_fixed_container(
	load_equipment_metadata_df, container_name: str='40FT'
):
	"""
	Build fixed 40FT container.
	
	Returns
	-------
	Container
	"""

	equipment_row = (
		load_equipment_metadata_df
		.loc[
			load_equipment_metadata_df[
				"equipment_name"
			]
			.str.upper()
			.str.contains(
				container_name,
				na=False
			)
		]
		.iloc[0]
	)

	container = Container(

		containerId=str(
			equipment_row["equipment_id"]
		),

		containerType=
			equipment_row["equipment_name"],

		depth=
			equipment_row["length_mm"],

		width=
			equipment_row["width_mm"],

		height=
			equipment_row["height_mm"],

		internalDepth=
			equipment_row["internal_length_mm"],

		internalWidth=
			equipment_row["internal_width_mm"],

		internalHeight=
			equipment_row["internal_height_mm"],

		maxPayloadWeightIn_kg=
			equipment_row["max_payload_weight_kg"],

		tareWeightIn_kg=
			equipment_row["tare_weight_kg"],

		maxVolume_m3=(

			equipment_row["internal_length_mm"]
			*
			equipment_row["internal_width_mm"]
			*
			equipment_row["internal_height_mm"]

		) / 1_000_000_000,

		doorWidth=
			equipment_row["door_width_mm"],

		doorHeight=
			equipment_row["door_height_mm"],

		pallets=[],
		axles=[
			Axle(
				axleId="FRONT_ZONE",
				maxWeightIn_kg=16000,
				positionXFromFrontIn_mm=3000
			),
			Axle(
				axleId="REAR_ZONE",
				maxWeightIn_kg=24000,
				positionXFromFrontIn_mm=10000
			)
		]
	)

	return container


In [29]:

container = build_fixed_container(load_equipment_metadata_df=load_equipment_metadata_df)

In [15]:
# Step 1 - Build Daily Demand
def build_daily_demand(
    shipment_plans_df: pd.DataFrame,
    planning_date
):

    df = shipment_plans_df.copy()

    planning_date = pd.to_datetime(
        planning_date
    )

    df["estimated_delivery_date"] = pd.to_datetime(
        df["estimated_delivery_date"],
        errors="coerce"
    )

    df = df[
        df["estimated_delivery_date"]
        <= planning_date
    ]

    df = df[
        df["planned_quantity"] > 0
    ]

    df["remaining_quantity"] = (
        df["planned_quantity"]
        -
        df["shipped_quantity"].fillna(0)
    )

    df = df[
        df["remaining_quantity"] > 0
    ]

    return df.reset_index(
        drop=True
    )

In [16]:
# Step 2 - Convert Units To Pallets

def convert_units_to_pallets(
    daily_demand_df
):

    df = daily_demand_df.copy()

    df["required_pallets"] = (
        df["remaining_quantity"]
        /
        df["unit_count_in_pallet"]
    )

    return df

In [17]:
# Step 3 - Apply Partial Pallet Rules

def apply_partial_pallet_rules(
    pallet_df,
    round_up_threshold=0.6
):

    df = pallet_df.copy()

    df["full_pallets"] = np.floor(
        df["required_pallets"]
    )

    df["fractional_pallet"] = (
        df["required_pallets"]
        -
        df["full_pallets"]
    )

    df["rounded_pallets"] = np.where(
        df["fractional_pallet"]
        >= round_up_threshold,
        np.ceil(
            df["required_pallets"]
        ),
        np.floor(
            df["required_pallets"]
        )
    )

    return df

In [18]:
# Step 4 - Build Shipment Buckets
def build_shipment_buckets(
    shipment_df
):

    bucket_df = (
        shipment_df
        .groupby(
            [
                "estimated_delivery_date",
                "origin_location_id",
                "destination_location_id",
                "sku_id"
            ],
            as_index=False
        )

        .agg(
            {
                "rounded_pallets": "sum",
                "pallet_weight_in_kg": "min",
                "item_weight_in_kg": "min",
                "unit_count_in_pallet": 'min',
                "priority": "max"
            }
        )
    )

    return bucket_df

In [19]:
# Step 5 - Build Load Queue

def build_load_queue(
    bucket_df
):

    return (
        bucket_df
        .sort_values(
            [
                "estimated_delivery_date",
                "priority",
                "rounded_pallets"
            ],
            ascending=[
                True,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

In [20]:
# Step 7 - Container Build Solver

def solve_container_build(
    load_queue_df,
    container: Container
):

    container_builds = []
    container_number = 1
    current_weight = 0
    current_pallets = []

    max_weight = (
        container.maxPayloadWeight
    )

    for _, row in load_queue_df.iterrows():
        # Issue: All Pallets are considered regardless of the limit 
        # Fix 1: Divide the pallets into considerable chunks and actively manage weight of the container being loaded
        # Fix 2: Also track the container area occupancy for each of the pallets that are getting loaded
        
        pallet_weight = (row['pallet_weight_in_kg'] * row["rounded_pallets"])

        pallet_count = int(
            row["rounded_pallets"]
        )

        print("container_id: " f"CONT_{container_number}", "len of pallet: " f"{pallet_count}", "max_weight: " f"{max_weight}", "pallet_weight: " f"{pallet_weight}")
        for _ in range(
            pallet_count
        ):

            if ((current_weight  + pallet_weight) > max_weight):
                # 
                container_builds.append({
                    "container_id":
                        f"CONT_{container_number}",
                    "pallets":
                        current_pallets
                })
                
                # Moves to build next container
                container_number += 1
                current_pallets = []
                current_weight = 0

            current_pallets.append({
                "sku_id":
                    row["sku_id"],

                "weight_kg":
                    pallet_weight,

                "destination_location_id":
                    row["destination_location_id"]
            })

            current_weight += (
                pallet_weight
            )

    if current_pallets:

        container_builds.append({

            "container_id":

                f"CONT_{container_number}",

            "pallets":

                current_pallets
        })

    return container_builds


In [21]:
# Step 8 - Build Physical Pallets

def build_pallets(
    container_builds,
    shipment_plans_df
):

    pallets = []

    for container in container_builds:

        for pallet in container["pallets"]:

            sku = pallet["sku_id"]

            shipment_row = (
                shipment_plans_df
                .loc[
                    shipment_plans_df[
                        "sku_id"
                    ]
                    == sku
                ]
                .iloc[0]
            )

            pallets.append({
                "container_id":
                    container[
                        "container_id"
                    ],

                "pallet_id":

                    str(
                        uuid.uuid4()
                    ),

                "sku_id":
                    sku,

                "weight_kg":
                    pallet[
                        "weight_kg"
                    ],

                "length_mm":
                    shipment_row[
                        "pallet_length_mm"
                    ],

                "width_mm":
                    shipment_row[
                        "pallet_width_mm"
                    ],

                "height_mm":
                    shipment_row[
                        "pallet_height_mm"
                    ],

                "position_x": 0,

                "position_y": 0,

                "position_z": 0
            })

    return pd.DataFrame(
        pallets
    )




In [22]:
# Step 9 - Place Pallets


def place_pallets(
    pallet_df,
    container: Container
):

    df = pallet_df.copy()

    current_x = 0
    current_z = 0

    row_depth = 0

    container_length = (
        container.internalDepth
        
    )

    container_width = (
        container.internalWidth
        
    )

    for idx in df.index:

        pallet_length = (
            df.loc[
                idx,
                "length_mm"
            ]
        )

        pallet_width = (
            df.loc[
                idx,
                "width_mm"
            ]
        )

        if (

            current_z
            + pallet_width

            >

            container_width

        ):

            current_x += row_depth

            current_z = 0

            row_depth = 0

        if (

            current_x
            + pallet_length
            >
            container_length

        ):
            pass
            # raise Exception(
            #     "Container full push to next container. Fix: Weight / volume prior selection"
            # )

        df.loc[
            idx,
            "position_x"
        ] = current_x

        df.loc[
            idx,
            "position_z"
        ] = current_z

        current_z += pallet_width

        row_depth = max(
            row_depth,
            pallet_length
        )

    return df

In [23]:
# Step 10 - Center of Gravity

def calculate_center_of_gravity(
    pallet_df
):

    total_weight = (
        pallet_df[
            "weight_kg"
        ].sum()
    )

    cg_x = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_x"
        ]

    ).sum() / total_weight

    cg_z = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_z"
        ]

    ).sum() / total_weight

    return {

        "cg_x": cg_x,

        "cg_z": cg_z
    }

In [ ]:
daily_demand_df = build_daily_demand(...)
pallet_candidate_df = convert_units_to_pallets(...)
pallet_candidate_df = apply_partial_pallet_rules(...)
pallet_candidate_df = calculate_pallet_metrics(...)
fill_candidates_df = build_container_fill_candidates(...)
allocated_df, pending_df = allocate_pallets_to_container(
        fill_candidates_df,
        container
    )

backlog_df = build_pending_backlog(
        pending_df
    )

physical_pallet_df = build_pallets(
        allocated_df
    )

positioned_pallet_df = place_pallets(
        physical_pallet_df,
        container
    )

In [46]:
# Run Pipeline

planning_date = "2026-06-10"

daily_demand_df = build_daily_demand(
    updated_shipment_plans_df,
    planning_date
)

pallet_req_df = convert_units_to_pallets(
    daily_demand_df
)

shipment_bucket_df = (
    apply_partial_pallet_rules(
        pallet_req_df
    )
)

bucket_df = build_shipment_buckets(
    shipment_bucket_df
)

load_queue_df = build_load_queue(
    bucket_df
)

container = build_fixed_container(
    load_equipment_metadata_df
)

container_builds = (
    solve_container_build(
        load_queue_df,
        container
    )
)


container_id: CONT_1 len of pallet: 132 max_weight: 25000.0 pallet_weight: 102923.04000000001
container_id: CONT_133 len of pallet: 78 max_weight: 25000.0 pallet_weight: 51796.67999999999
container_id: CONT_211 len of pallet: 72 max_weight: 25000.0 pallet_weight: 52809.12
container_id: CONT_283 len of pallet: 30 max_weight: 25000.0 pallet_weight: 21432.3
container_id: CONT_313 len of pallet: 29 max_weight: 25000.0 pallet_weight: 22611.88
container_id: CONT_342 len of pallet: 28 max_weight: 25000.0 pallet_weight: 25744.039999999997
container_id: CONT_370 len of pallet: 28 max_weight: 25000.0 pallet_weight: 15647.240000000002
container_id: CONT_398 len of pallet: 22 max_weight: 25000.0 pallet_weight: 12853.060000000001
container_id: CONT_420 len of pallet: 21 max_weight: 25000.0 pallet_weight: 18136.44
container_id: CONT_441 len of pallet: 17 max_weight: 25000.0 pallet_weight: 9507.76
container_id: CONT_450 len of pallet: 16 max_weight: 25000.0 pallet_weight: 5704.32
container_id: CONT_4

In [43]:
len(container_builds)

245779

In [51]:
container_builds[312]

{'container_id': 'CONT_313',
 'pallets': [{'sku_id': '186792',
   'weight_kg': 21432.3,
   'destination_location_id': '6017'}]}

In [27]:
len(container_builds[:5])

5

In [55]:

pallet_df = build_pallets(
    container_builds[:5],
    updated_shipment_plans_df
)

pallet_df = place_pallets(
    pallet_df,
    container
)

cg = calculate_center_of_gravity(
    pallet_df
)

print(cg)


{'cg_x': np.float64(609.5), 'cg_z': np.float64(508.00000000000006)}


In [54]:
container_builds

[{'container_id': 'CONT_1', 'pallets': []},
 {'container_id': 'CONT_2',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_3',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_4',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_5',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_6',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_7',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    'destination_location_id': '6062'}]},
 {'container_id': 'CONT_8',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 102923.04000000001,
    

In [ ]:
from copy import deepcopy

def build_container_objects(
    container_builds,
    pallet_position_df,
    container_template
):
    """
    Build Pydantic Container objects.

    ```
    Returns
    -------
    List[Container]
    """

    containers = []

    for build in container_builds:

        container_id = build["container_id"]

        container_pallets = (
            pallet_position_df
            [
                pallet_position_df[
                    "container_id"
                ]
                ==
                container_id
            ]
        )

        container = deepcopy(
            container_template
        )

        container.containerId = (
            container_id
        )

        container.pallets = []

        total_weight = 0
        total_volume = 0

        for _, pallet_row in (

            container_pallets

            .iterrows()
        ):

            pallet = Pallet(

                label=str(
                    pallet_row[
                        "sku_id"
                    ]
                ),

                color="#4CAF50",

                dimensions=Dimension(

                    depth=int(
                        pallet_row[
                            "length_mm"
                        ]
                    ),

                    width=int(
                        pallet_row[
                            "width_mm"
                        ]
                    ),

                    height=int(
                        pallet_row[
                            "height_mm"
                        ]
                    )
                ),

                position=Position(

                    x=int(
                        pallet_row[
                            "position_x"
                        ]
                    ),

                    y=int(
                        pallet_row[
                            "position_y"
                        ]
                    ),

                    z=int(
                        pallet_row[
                            "position_z"
                        ]
                    )
                )
            )

            container.pallets.append(
                pallet
            )

            total_weight += (
                pallet_row[
                    "weight_kg"
                ]
            )

            # total_volume += (
            #     pallet_row[
            #         "volume_m3"
            #     ]
            # )

        container.summary.totalPallets = (
            len(
                container.pallets
            )
        )

        container.summary.totalWeight = (
            round(
                total_weight,
                2
            )
        )

        container.summary.totalVolume = (
            round(
                total_volume,
                2
            )
        )

        containers.append(
            container
        )

    return containers


In [64]:
containers = build_container_objects(
    container_builds=container_builds[:315],
    pallet_position_df=pallet_df,
    container_template=container
)

In [67]:
# for _container_ in containers:
# for _container_ in containers:
container_visualization(data=containers[1])
container_visualization(data=containers[2])
container_visualization(data=containers[3])

http://localhost:5173/container-visualization?data=%7B%22containerId%22%3A%20%22CONT_2%22%2C%20%22containerType%22%3A%20%22CONTAINER%2040FT%20GP%22%2C%20%22depth%22%3A%2012192.0%2C%20%22width%22%3A%202438.0%2C%20%22height%22%3A%202591.0%2C%20%22internal_depth%22%3A%2012031.0%2C%20%22internal_width%22%3A%202352.0%2C%20%22internal_height%22%3A%202393.0%2C%20%22maxPayloadWeight%22%3A%2025000.0%2C%20%22tareWeight%22%3A%200.0%2C%20%22maxVolume%22%3A%2067.714510416%2C%20%22unit%22%3A%20%22mm%22%2C%20%22door_width%22%3A%202340.0%2C%20%22door_height%22%3A%202280.0%2C%20%22axles%22%3A%20%5B%7B%22axleId%22%3A%20%22DEFAULT%22%2C%20%22maxWeight%22%3A%2030000%2C%20%22positionX%22%3A%201371.6%7D%5D%2C%20%22pallets%22%3A%20%5B%7B%22dimensions%22%3A%20%7B%22depth%22%3A%201219%2C%20%22width%22%3A%201016%2C%20%22height%22%3A%201277%7D%2C%20%22position%22%3A%20%7B%22x%22%3A%200%2C%20%22y%22%3A%200%2C%20%22z%22%3A%200%7D%2C%20%22label%22%3A%20%22203277%22%2C%20%22color%22%3A%20%22%234CAF50%22%7D%5D%2C%20%

http://localhost:5173/container-visualization?data=%7B%22containerId%22%3A%20%22CONT_3%22%2C%20%22containerType%22%3A%20%22CONTAINER%2040FT%20GP%22%2C%20%22depth%22%3A%2012192.0%2C%20%22width%22%3A%202438.0%2C%20%22height%22%3A%202591.0%2C%20%22internal_depth%22%3A%2012031.0%2C%20%22internal_width%22%3A%202352.0%2C%20%22internal_height%22%3A%202393.0%2C%20%22maxPayloadWeight%22%3A%2025000.0%2C%20%22tareWeight%22%3A%200.0%2C%20%22maxVolume%22%3A%2067.714510416%2C%20%22unit%22%3A%20%22mm%22%2C%20%22door_width%22%3A%202340.0%2C%20%22door_height%22%3A%202280.0%2C%20%22axles%22%3A%20%5B%7B%22axleId%22%3A%20%22DEFAULT%22%2C%20%22maxWeight%22%3A%2030000%2C%20%22positionX%22%3A%201371.6%7D%5D%2C%20%22pallets%22%3A%20%5B%7B%22dimensions%22%3A%20%7B%22depth%22%3A%201219%2C%20%22width%22%3A%201016%2C%20%22height%22%3A%201277%7D%2C%20%22position%22%3A%20%7B%22x%22%3A%200%2C%20%22y%22%3A%200%2C%20%22z%22%3A%201016%7D%2C%20%22label%22%3A%20%22203277%22%2C%20%22color%22%3A%20%22%234CAF50%22%7D%5D%2C%

http://localhost:5173/container-visualization?data=%7B%22containerId%22%3A%20%22CONT_4%22%2C%20%22containerType%22%3A%20%22CONTAINER%2040FT%20GP%22%2C%20%22depth%22%3A%2012192.0%2C%20%22width%22%3A%202438.0%2C%20%22height%22%3A%202591.0%2C%20%22internal_depth%22%3A%2012031.0%2C%20%22internal_width%22%3A%202352.0%2C%20%22internal_height%22%3A%202393.0%2C%20%22maxPayloadWeight%22%3A%2025000.0%2C%20%22tareWeight%22%3A%200.0%2C%20%22maxVolume%22%3A%2067.714510416%2C%20%22unit%22%3A%20%22mm%22%2C%20%22door_width%22%3A%202340.0%2C%20%22door_height%22%3A%202280.0%2C%20%22axles%22%3A%20%5B%7B%22axleId%22%3A%20%22DEFAULT%22%2C%20%22maxWeight%22%3A%2030000%2C%20%22positionX%22%3A%201371.6%7D%5D%2C%20%22pallets%22%3A%20%5B%7B%22dimensions%22%3A%20%7B%22depth%22%3A%201219%2C%20%22width%22%3A%201016%2C%20%22height%22%3A%201277%7D%2C%20%22position%22%3A%20%7B%22x%22%3A%201219%2C%20%22y%22%3A%200%2C%20%22z%22%3A%200%7D%2C%20%22label%22%3A%20%22203277%22%2C%20%22color%22%3A%20%22%234CAF50%22%7D%5D%2C%

In [59]:
containers[1].pallets

[Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=0, y=0, z=0), label='203277', color='#4CAF50')]

## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 